# 03 — Fine-tuning NAFNet sur les plaques FR

Ce notebook fine-tune NAFNet (SOTA deblurring) sur notre dataset de plaques francaises.

**Input :** `dataset/` (50K paires floue → nette du notebook 02)
**Output :** `checkpoints/nafnet_plates_fr.pth` (poids du modele)

**Runtime : GPU T4 requis (Google Colab gratuit), ~2-4h**

In [ ]:
# Verifier le GPU
!nvidia-smi
import torch
print(f"CUDA disponible : {torch.cuda.is_available()}")
print(f"GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Aucun'}")

In [ ]:
!pip install -q torch torchvision pillow scikit-image tqdm matplotlib

# Cloner NAFNet
!git clone -q https://github.com/megvii-research/NAFNet.git
!pip install -q -e NAFNet/

# Telecharger les poids pre-entraines (GoPro deblurring)
!mkdir -p checkpoints
!wget -q https://github.com/megvii-research/NAFNet/releases/download/v0.0.1/NAFNet-GoPro-width64.pth \
    -O checkpoints/NAFNet-GoPro-width64.pth
print('NAFNet pret')

In [ ]:
# Charger le dataset depuis Google Drive
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p dataset
!cp -r /content/drive/MyDrive/deblurring_dataset/* dataset/

import json
with open('dataset/train.json') as f: train_pairs = json.load(f)
with open('dataset/val.json') as f: val_pairs = json.load(f)
print(f"Train: {len(train_pairs)} paires, Val: {len(val_pairs)} paires")

## 1. Dataset PyTorch

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

class PlateDeblurDataset(Dataset):
    """Dataset de paires (image degradee, image nette)."""

    def __init__(self, pairs, clean_dir='dataset/clean', degraded_dir='dataset/degraded'):
        self.pairs = pairs
        self.clean_dir = clean_dir
        self.degraded_dir = degraded_dir
        self.transform = transforms.Compose([
            transforms.ToTensor(),  # [0, 255] → [0, 1]
        ])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        degraded = Image.open(f"{self.degraded_dir}/{pair['degraded']}").convert('RGB')
        clean = Image.open(f"{self.clean_dir}/{pair['clean']}").convert('RGB')
        return self.transform(degraded), self.transform(clean)


# Creer les dataloaders
train_dataset = PlateDeblurDataset(train_pairs)
val_dataset = PlateDeblurDataset(val_pairs)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

# Verifier
degraded, clean = next(iter(train_loader))
print(f"Batch shape: degraded={degraded.shape}, clean={clean.shape}")
# → [8, 3, 110, 520]

## 2. Charger NAFNet pre-entraine

In [ ]:
import sys
sys.path.insert(0, 'NAFNet')
from basicsr.models.archs.NAFNet_arch import NAFNet

# NAFNet-width64 (le modele GoPro)
model = NAFNet(
    img_channel=3,
    width=64,
    middle_blk_num=12,
    enc_blk_nums=[2, 2, 4, 8],
    dec_blk_nums=[2, 2, 2, 2],
)

# Charger les poids pre-entraines
state_dict = torch.load('checkpoints/NAFNet-GoPro-width64.pth', map_location='cpu')
# Les poids NAFNet sont parfois wrapes dans 'params' ou 'state_dict'
if 'params' in state_dict:
    state_dict = state_dict['params']
elif 'state_dict' in state_dict:
    state_dict = state_dict['state_dict']

model.load_state_dict(state_dict, strict=False)
model = model.cuda()

n_params = sum(p.numel() for p in model.parameters())
print(f"NAFNet charge : {n_params / 1e6:.1f}M parametres")

## 3. Entrainement

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
import matplotlib.pyplot as plt

# Config
EPOCHS = 50
LR = 1e-4
DEVICE = 'cuda'

criterion = nn.L1Loss()  # L1 loss (standard pour la restauration d'image)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# Historique
history = {'train_loss': [], 'val_loss': [], 'val_psnr': []}
best_val_loss = float('inf')

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as calc_psnr

def validate(model, val_loader):
    """Calcule la loss et le PSNR sur le val set."""
    model.eval()
    total_loss, total_psnr, n = 0, 0, 0
    with torch.no_grad():
        for degraded, clean in val_loader:
            degraded, clean = degraded.cuda(), clean.cuda()
            restored = model(degraded)
            loss = criterion(restored, clean)
            total_loss += loss.item() * degraded.size(0)

            # PSNR
            for i in range(degraded.size(0)):
                r = restored[i].cpu().numpy().transpose(1, 2, 0)
                c = clean[i].cpu().numpy().transpose(1, 2, 0)
                total_psnr += calc_psnr(c, r, data_range=1.0)
            n += degraded.size(0)

    return total_loss / n, total_psnr / n

In [ ]:
# Boucle d'entrainement
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    n_batches = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for degraded, clean in pbar:
        degraded, clean = degraded.cuda(), clean.cuda()

        optimizer.zero_grad()
        restored = model(degraded)
        loss = criterion(restored, clean)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    scheduler.step()

    # Validation
    val_loss, val_psnr = validate(model, val_loader)

    history['train_loss'].append(epoch_loss / n_batches)
    history['val_loss'].append(val_loss)
    history['val_psnr'].append(val_psnr)

    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss/n_batches:.4f} | Val Loss: {val_loss:.4f} | Val PSNR: {val_psnr:.2f} dB")

    # Sauvegarder le meilleur modele
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'checkpoints/nafnet_plates_fr_best.pth')
        print(f"  → Meilleur modele sauvegarde (PSNR: {val_psnr:.2f} dB)")

# Sauvegarder le dernier
torch.save(model.state_dict(), 'checkpoints/nafnet_plates_fr_last.pth')
print("\nEntrainement termine !")

## 4. Courbes d'entrainement

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train', color='#3b82f6')
ax1.plot(history['val_loss'], label='Val', color='#f59e0b')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('L1 Loss')
ax1.set_title('Loss')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(history['val_psnr'], color='#10b981')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('PSNR (dB)')
ax2.set_title('PSNR sur le val set')
ax2.grid(alpha=0.3)

plt.suptitle('Entrainement NAFNet — Plaques FR', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 5. Visualiser les resultats

In [ ]:
# Charger le meilleur modele
model.load_state_dict(torch.load('checkpoints/nafnet_plates_fr_best.pth'))
model.eval()

# Quelques exemples
fig, axes = plt.subplots(4, 3, figsize=(15, 14))
test_batch = next(iter(val_loader))

with torch.no_grad():
    restored = model(test_batch[0].cuda()).cpu()

for i in range(4):
    deg = test_batch[0][i].permute(1, 2, 0).numpy()
    cln = test_batch[1][i].permute(1, 2, 0).numpy()
    res = restored[i].permute(1, 2, 0).numpy().clip(0, 1)

    axes[i][0].imshow(deg)
    axes[i][0].set_title('Degradee (input)', fontsize=10)
    axes[i][0].axis('off')

    axes[i][1].imshow(res)
    p = calc_psnr(cln, res, data_range=1.0)
    axes[i][1].set_title(f'Restauree (PSNR: {p:.1f} dB)', fontsize=10)
    axes[i][1].axis('off')

    axes[i][2].imshow(cln)
    axes[i][2].set_title('Verite terrain', fontsize=10)
    axes[i][2].axis('off')

plt.suptitle('Resultats : Degradee → Notre modele → Verite terrain', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results_deblur.png', dpi=150)
plt.show()

## 6. Exporter pour le deploiement

In [ ]:
# Sauvegarder le modele complet (architecture + poids)
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'img_channel': 3,
        'width': 64,
        'middle_blk_num': 12,
        'enc_blk_nums': [2, 2, 4, 8],
        'dec_blk_nums': [2, 2, 2, 2],
    },
    'history': history,
    'best_val_psnr': max(history['val_psnr']),
}, 'checkpoints/nafnet_plates_fr_full.pth')

print(f"Modele exporte : checkpoints/nafnet_plates_fr_full.pth")
print(f"Meilleur PSNR : {max(history['val_psnr']):.2f} dB")

# Sauvegarder sur Google Drive
!mkdir -p /content/drive/MyDrive/deblurring_model
!cp checkpoints/nafnet_plates_fr_best.pth /content/drive/MyDrive/deblurring_model/
!cp checkpoints/nafnet_plates_fr_full.pth /content/drive/MyDrive/deblurring_model/
!cp training_curves.png results_deblur.png /content/drive/MyDrive/deblurring_model/
print("Modele + graphiques sauvegardes sur Drive dans /deblurring_model/")

## Next

Le modele est entraine. Passe au notebook **04_evaluate.ipynb** pour le benchmark complet contre Real-ESRGAN et NAFNet vanilla.